This Notebook comes from our OpenEO examples gallery, see https://github.com/geojupyter/openeo-gallery for more.

It was adapted to use the "openeo.ds.io" backend with CDSE auth.

## Prerequisites

**This Notebook only runs in JupyterLab (not JupyterLite due to CORS issues with CDSE auth).**

## License

This notebook is an adaptation of the openEO-UDP notebook
[ndci_cyanobacteria.ipynb](https://github.com/developmentseed/openeo-udp/blob/main/notebooks/sentinel/sentinel-2/marine_and_water_bodies/ndci_cyanobacteria.ipynb)
by Development Seed. That notebook is a conversion of a Sentinel Hub evalscript and is licensed under **CC-BY-SA-4.0**.
This adaptation is also licensed under **CC-BY-SA-4.0**.

- Original evalscript: https://custom-scripts.sentinel-hub.com/sentinel-2/cyanobacteria_chla_ndci_l1c/
- Source: Sentinel Hub Custom Scripts (CC-BY-SA-4.0)
- Conversion: Development Seed (openEO-UDP project)

# Cyanobacteria chlorophyll-a (NDCI) over the Venice lagoon, June 2026

This notebook calculates chlorophyll-a in the Venice lagoon from Sentinel-2 L2A data with openEO.
It shows the result as a dynamic tile layer in a JupyterGIS map, on top of an OpenStreetMap basemap.

The steps are:
1. Connect to the openEO backend.
2. Load the Sentinel-2 L2A scenes of the Venice lagoon from June 2026 and make a first-valid-pixel mosaic.
3. Find the water pixels with water indices (MNDWI, NDWI, AWEI and others).
4. Calculate the Floating Algae Index (FAI) to find surface blooms.
5. Calculate the Normalized Difference Chlorophyll Index (NDCI) and the chlorophyll-a concentration.
6. Apply a color map and show the result in JupyterGIS.

**NDCI = (B05 - B04) / (B05 + B04)**

**Chl-a (μg/L) = 826.57 × NDCI³ - 176.43 × NDCI² + 19 × NDCI + 4.071**

The model is calibrated on synthetic data for the cyanobacteria *Microcystis aeruginosa*.

In [ ]:
import openeo
from openeo.processes import array_create, if_, process

connection = openeo.connect("openeo.ds.io")
connection.authenticate_oidc(client_id="openeo-developmentseed")

## Parameters

The bounding box covers the full Venice lagoon, from the north lagoon to Chioggia.

The time range is June 2026. The end date is exclusive in openEO.
The cloud cover filter keeps only scenes with 10 % cloud cover or less.

In [ ]:
location_name = "Venice lagoon, Italy"

bounding_box = {"west": 12.15, "south": 45.18, "east": 12.60, "north": 45.58}
temporal_extent = ["2026-06-01", "2026-07-01"]
max_cloud_cover = 10

# Sentinel-2 bands for the algorithm, in this order:
# B02 (blue), B03 (green), B04 (red), B05 (red edge 705 nm), B07 (red edge 783 nm),
# B08 (NIR), B8A (narrow NIR), B11 (SWIR 1610 nm), B12 (SWIR 2190 nm)
bands = ["B02_20m", "B03_20m", "B04_20m", "B05_20m", "B07_20m", "B08_10m", "B8A_20m", "B11_20m", "B12_20m"]

## Load the Sentinel-2 data

This backend gives the `sentinel-2-l2a` bands as surface reflectance (0 to 1). The scale and offset are already applied.
Two Sentinel-2 orbits pass over the lagoon, and the swath edge of one orbit crosses the western lagoon.
The no-data border of a scene has reflectance -0.1, but it is not masked. The algorithm would see it as water with NDCI = 0 and paint it in a false color.

The notebook does two steps to prevent this:
1. It masks the no-data pixels with the scene classification band (`SCL` = 0).
2. It reduces the time dimension with the backend process `firstpixel`. For each pixel, `firstpixel` keeps the first valid value and fills masked pixels from the next scene.
   The standard `first` reducer does not do this. It keeps the value of the first scene, also when that value is masked.

The result is a mosaic of several June dates. You can see a seam where two scenes meet.

In [ ]:
s2cube = connection.load_collection(
    "sentinel-2-l2a",
    spatial_extent=bounding_box,
    temporal_extent=temporal_extent,
    bands=[*bands, "SCL_20m"],
    properties={"eo:cloud_cover": lambda x: x <= max_cloud_cover},
)

# Mask the no-data pixels (SCL = 0), then remove the SCL band
nodata = s2cube.band("SCL_20m") == 0
s2cube = s2cube.filter_bands(bands).mask(nodata)

# For each pixel, keep the first valid (non-masked) value in June
s2cube = s2cube.reduce_dimension(dimension="t", reducer="firstpixel")

## Calculate NDCI and the chlorophyll-a color map

The color map follows the original script:
- **Blue**: low concentration (< 5 μg/L)
- **Green**: medium concentration (5 to 38 μg/L)
- **Yellow**: high concentration (38 to 150 μg/L)
- **Orange to red**: very high concentration (> 150 μg/L)

Land pixels show true color. Pixels with floating algae (FAI > 0.08) show red.

In [ ]:
def cyanobacteria_chl_a_visualization(data):
    """CyanoLakes chlorophyll-a visualization.

    Input array: [B02, B03, B04, B05, B07, B08, B8A, B11, B12] as reflectance (0 to 1).
    Output array: [R, G, B] in the range 0 to 1.
    """
    B02, B03, B04, B05, B07, B08, B8A, B11, B12 = (data[i] for i in range(9))

    MNDWI_threshold = 0.42
    NDWI_threshold = 0.4

    # Water body identification
    MNDWI_val = (B03 - B11) / (B03 + B11)
    NDWI_val = (B03 - B08) / (B03 + B08)
    NDVI_val = (B08 - B04) / (B08 + B04)
    NDWI_leaves = (B08 - B11) / (B08 + B11)
    AWEISH = B02 + 2.5 * B03 - 1.5 * (B08 + B11) - 0.25 * B12
    AWEINSH = 4 * (B03 - B11) - (0.25 * B08 + 2.75 * B11)
    DBSI = ((B11 - B03) / (B11 + B03)) - NDVI_val

    water = if_(
        MNDWI_val > MNDWI_threshold,
        1,
        if_(
            NDWI_val > NDWI_threshold,
            1,
            if_(
                AWEINSH > 0.1879,
                1,
                if_(
                    AWEISH > 0.1112,
                    1,
                    if_(NDVI_val < -0.2, 1, if_(NDWI_leaves > 1, 1, 0)),
                ),
            ),
        ),
    )
    # Remove urban areas and bare soil from the water pixels
    water = if_(
        water == 1,
        if_(AWEINSH <= -0.03, 0, if_(DBSI > 0, 0, water)),
        water,
    )

    # Floating Algae Index
    FAI_val = B07 - (B04 + (B8A - B04) * ((783.0 - 665.0) / (865.0 - 665.0)))

    # NDCI and chlorophyll-a
    NDCI_val = (B05 - B04) / (B05 + B04)
    chl = 826.57 * (NDCI_val**3) - 176.43 * (NDCI_val**2) + 19 * NDCI_val + 4.071

    ones = B04 * 0 + 1

    def rgb(r, g, b):
        return array_create([ones * (r / 255), ones * (g / 255), ones * (b / 255)])

    land_color = array_create([B04 * 3, B03 * 3, B02 * 3])
    red_bloom = rgb(233, 72, 21)

    # (upper limit in Î¼g/L, color); the last color is used above the last limit
    classes = [
        (0.5, rgb(0, 0, 255)),
        (2.5, rgb(0, 59, 255)),
        (5, rgb(15, 113, 141)),
        (8, rgb(13, 141, 103)),
        (14, rgb(42, 226, 28)),
        (24, rgb(134, 247, 0)),
        (38, rgb(208, 240, 0)),
        (75, rgb(248, 207, 2)),
        (150, rgb(240, 159, 8)),
        (350, rgb(239, 101, 15)),
    ]
    chl_color = rgb(233, 72, 21)
    for limit, color in reversed(classes):
        chl_color = if_(chl < limit, color, chl_color)

    return if_(water == 0, land_color, if_(FAI_val > 0.08, red_bloom, chl_color))


chl_a_image = s2cube.apply_dimension(
    dimension="spectral",
    process=cyanobacteria_chl_a_visualization,
)

# Scale to 0-255 (values outside 0-1 are clipped) and make integers for the PNG tiles
chl_a_image = chl_a_image.linear_scale_range(0, 1, 0, 255).apply(
    lambda x: process("trunc", x=x),
)

graph = chl_a_image

## Show the result in JupyterGIS

The map has an OpenStreetMap basemap. The openEO layer is added on top of it.
`add_openeo_tile_layer` creates an XYZ service on the backend and fits the map to the bounding box.

The backend calculates each tile on request from several scenes. A tile can take 10 to 25 seconds, so the layer fills in slowly.

In [ ]:
from jupytergis import GISDocument

doc = GISDocument(latitude=45.38, longitude=12.37, zoom=11)

In [ ]:
doc.add_raster_layer(
    url="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="OpenStreetMap",
    attribution="Â© OpenStreetMap contributors",
)

In [ ]:
doc.add_openeo_tile_layer(graph, name="NDCI chlorophyll-a (June 2026)")
doc

## Interpretation guide

| Color | Chl-a (μg/L) | Water quality status |
|-------|--------------|----------------------|
| Deep blue | < 0.5 | Ultra-oligotrophic (very clear) |
| Blue | 0.5 to 2.5 | Oligotrophic (clear) |
| Blue-cyan | 2.5 to 8 | Mesotrophic (moderate) |
| Green | 8 to 24 | Eutrophic (nutrient-rich) |
| Yellow | 24 to 75 | Highly eutrophic |
| Orange | 75 to 150 | Hypereutrophic |
| Red | > 150 or FAI > 0.08 | Severe bloom or floating algae |

### Limitations

1. The model is best for blooms where cyanobacteria are dominant.
2. The Venice lagoon is shallow and turbid. Suspended sediment and a visible bottom can bias the NDCI.
3. The model was calibrated for fresh water. The lagoon is brackish, so use the values as relative, not absolute.
4. Surface blooms can saturate the signal.
5. Mixed phytoplankton populations can decrease the accuracy.
6. The mosaic mixes dates. Chlorophyll can change from one date to the next, so values can jump at a scene seam.
7. Thin haze can make the water test fail. These water pixels then show in true color, as dark patches.

### Citation

Kravitz, J & Matthews M., 2020. Chlorophyll-a for cyanobacteria blooms from Sentinel-2. CyanoLakes.

### References

- Hu, C. (2009). A novel ocean color index to detect floating algae in the global oceans. *Remote Sensing of Environment*, 113(10), 2118-2129.
- Kravitz, J., Matthews, M., Lain, L., Fawcett, S., & Bernard, S. (2021). Potential for high fidelity global mapping of common inland water quality products at high spatial and temporal resolutions based on a synthetic data and machine learning approach. *Frontiers in Environmental Science*, 19.
- Mishra, S., & Mishra, D. R. (2012). Normalized difference chlorophyll index: A novel model for remote estimation of chlorophyll-a concentration in turbid productive waters. *Remote Sensing of Environment*, 117, 394-406.

## Attribution

- **Original script:** [Cyanobacteria Chlorophyll-a (NDCI) - L1C](https://custom-scripts.sentinel-hub.com/sentinel-2/cyanobacteria_chla_ndci_l1c/)
- **Authors:** Johannah Kravitz, Mark Matthews (CyanoLakes)
- **Source repository:** [Sentinel Hub Custom Scripts](https://github.com/sentinel-hub/custom-scripts)
- **openEO conversion:** [Development Seed openEO-UDP](https://github.com/developmentseed/openeo-udp)